# Step 7 &mdash; Reproducible Deployment (Docker)

**Goal:** package the trained SwinV2-Tiny dual-stream classifier as a GPU-enabled Docker image whose predictions are **bit-for-bit identical** to local inference.

## Deployment scope

Distributing the full training pipeline is unnecessary for production. The deployment artifact only needs to take preprocessed inputs (tile + global images + manifest CSV) and produce per-class predictions. Reproducibility is verified by checksumming `predictions.csv` against a local-run baseline.

## 1. Artifacts shipped in the image

| File | Purpose |
|------|---------|
| `Dockerfile.gpu`             | Base PyTorch + CUDA image with pinned deps |
| `requirements.docker.txt`    | Locked package versions (timm, torchvision, sklearn, ...) |
| `infer.py`                   | Inference entry point |
| `best_model.pt`              | Final SwinV2-Tiny checkpoint |
| `val_final_thresholds.npy`   | Per-class thresholds (optional &mdash; falls back to fixed 0.5) |

## 2. Dockerfile

Minimal image based on the official PyTorch runtime. All packages are pinned to specific versions to guarantee reproducibility across machines and over time.

In [ ]:
DOCKERFILE = r"""
FROM pytorch/pytorch:2.9.1-cuda12.8-cudnn9-runtime

WORKDIR /app

RUN pip install --no-cache-dir --upgrade pip \
 && pip install --no-cache-dir \
      timm==1.0.22 \
      pandas==2.3.3 \
      pillow==12.0.0 \
      numpy==2.3.3 \
      torchvision==0.24.0 \
      scikit-learn==1.7.2

COPY infer.py /app/infer.py

CMD ["python", "infer.py", "-h"]
"""
print(DOCKERFILE)

## 3. Inference entry point (sketch)

`infer.py` rebuilds the SwinV2 dual-stream model, loads the checkpoint, runs the same preprocessing transforms used at training time, and emits a CSV of per-class probabilities + binary predictions.

In [ ]:
INFER_SKETCH = r"""
# infer.py (core flow)

ckpt = torch.load(args.ckpt, map_location=device)
label_cols = ckpt['label_cols']

model = SwinV2DualStream(
    backbone_name='swinv2_tiny_window8_256',
    num_classes=len(label_cols),
    pretrained=False,
).to(device)
model.load_state_dict(ckpt['model'])
model.eval()

df = pd.read_csv(args.csv)
ds = DualStreamInferenceDataset(
    df, Path(args.img_dir_tile), Path(args.img_dir_global), label_cols,
    tile_size=256, global_size=(256, 768),
)
loader = DataLoader(ds, batch_size=args.batch_size, shuffle=False)

probs = []
with torch.no_grad():
    for tile, global_img, _ in loader:
        logits = model(tile.to(device), global_img.to(device))
        probs.append(torch.sigmoid(logits).cpu().numpy())
probs = np.concatenate(probs)

thresholds = np.load(args.thresholds) if args.thresholds else 0.5
preds = (probs >= np.asarray(thresholds).reshape(1, -1)).astype(np.int32)

out = pd.DataFrame(probs, columns=[f'prob__{c}' for c in label_cols])
out = pd.concat([
    df[['filename_tile', 'filename_global', 'probe_id']].reset_index(drop=True),
    out,
    pd.DataFrame(preds, columns=[f'pred__{c}' for c in label_cols]),
], axis=1)
out.to_csv(args.out_csv, index=False)
"""
print(INFER_SKETCH)

## 4. Build & run

```bash
# Build the image
docker build -t wear-classifier:gpu -f Dockerfile.gpu .

# Run inference (mount the working dir into the container)
docker run --rm --gpus all \
  -v "$PWD:/app/work" \
  wear-classifier:gpu \
  python /app/infer.py \
    --ckpt        /app/work/best_model.pt \
    --thresholds  /app/work/val_final_thresholds.npy \
    --csv         /app/work/dataset_dualstream.csv \
    --img_dir_tile   /app/work/images_tiles \
    --img_dir_global /app/work/images_global \
    --out_csv     /app/work/predictions.docker.csv
```

The container is **stateless**: the checkpoint, thresholds, and data are injected via a bind mount, not baked into the image. This lets the same image serve any future retrained checkpoint without a rebuild.

## 5. Reproducibility check (local vs Docker)

After running both locally and inside Docker, the output CSVs should match byte-for-byte:

```bash
$ md5sum predictions.csv predictions.docker.csv
18d3a45584d42371ad12a1e65330840a  predictions.csv
18d3a45584d42371ad12a1e65330840a  predictions.docker.csv
```

Identical MD5 hashes confirm that:
- The Docker environment exactly reproduces the local inference behavior.
- Future redeployments will not silently change predictions due to library updates.

## 6. Why this matters in an industrial setting

- **Quality assurance teams** can validate predictions against the original model by comparing CSV checksums &mdash; no need to re-train or re-evaluate.
- **Operators** get a single, portable artifact that runs on any CUDA-12.8 host.
- **R&D / future iterations** can swap in retrained checkpoints (with the same label schema) without touching the deployment image.